# Fine-tuning de Qwen2.5-1.5B-Instruct para "Cholets Turístico El Alto"

Ajuste fino con **LoRA** (Low-Rank Adaptation) usando **Unsloth**, sobre el dataset generado con
`npm run dataset:generar` en el backend.

**Antes de empezar:** menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4** → Guardar.
Luego ejecuta las celdas en orden con ▶ (o *Entorno de ejecución → Ejecutar todas*).

Resultado: el archivo **`cholets-qwen.gguf`** para instalarlo en Ollama en tu PC.

## 1. Instalar Unsloth (2-3 minutos)

In [ ]:
%%capture
!pip install unsloth

## 2. Subir el dataset
Elige los archivos **`cholets_train.jsonl`** y **`cholets_val.jsonl`** de la carpeta `CODIGO/finetuning/dataset/`.

In [ ]:
from google.colab import files
subidos = files.upload()
assert 'cholets_train.jsonl' in subidos and 'cholets_val.jsonl' in subidos, 'Sube los dos archivos .jsonl'

## 3. Cargar el modelo base (Qwen2.5-1.5B-Instruct, cuantizado a 4 bits)

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_LEN = 2048  # los ejemplos mas largos tienen ~1.500 tokens
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length=MAX_LEN,
    load_in_4bit=True,
)

## 4. Configurar LoRA
Solo se entrenan matrices pequeñas añadidas a las capas de atención y MLP (~1 % de los parámetros);
el resto del modelo queda congelado.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                 # rango de las matrices LoRA
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
model.print_trainable_parameters()

## 5. Preparar el dataset con la plantilla de chat de Qwen

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")
ds = load_dataset("json", data_files={"train": "cholets_train.jsonl", "validation": "cholets_val.jsonl"})

def formatear(lote):
    return {"text": [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in lote["messages"]]}

ds = ds.map(formatear, batched=True)
print(ds)
print(ds["train"][0]["text"][-800:])

## 6. Entrenar (30-50 minutos en una T4)
Con `train_on_responses_only` el modelo aprende solo de las respuestas del asistente, no del contexto.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_LEN,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,   # lote efectivo de 8 ejemplos
        num_train_epochs=2,
        learning_rate=2e-4,
        warmup_steps=10,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        weight_decay=0.01,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        seed=3407,
        output_dir="salidas",
        report_to="none",
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
estadisticas = trainer.train()
print(f"Tiempo de entrenamiento: {estadisticas.metrics['train_runtime'] / 60:.1f} minutos")

## 7. Curva de pérdida (para la tesis)

In [ ]:
import matplotlib.pyplot as plt

historial = trainer.state.log_history
pasos_t = [h["step"] for h in historial if "loss" in h]
perdida_t = [h["loss"] for h in historial if "loss" in h]
pasos_v = [h["step"] for h in historial if "eval_loss" in h]
perdida_v = [h["eval_loss"] for h in historial if "eval_loss" in h]

plt.figure(figsize=(8, 4.5))
plt.plot(pasos_t, perdida_t, label="Entrenamiento")
plt.plot(pasos_v, perdida_v, "o-", label="Validación")
plt.xlabel("Paso")
plt.ylabel("Pérdida (loss)")
plt.title("Fine-tuning LoRA de Qwen2.5-1.5B · Cholets Turístico El Alto")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("curva_perdida.png", dpi=200)
plt.show()
files.download("curva_perdida.png")

## 8. Probar el modelo afinado

In [ ]:
import json
FastLanguageModel.for_inference(model)

ejemplos = [json.loads(l) for l in open("cholets_val.jsonl", encoding="utf-8")][:4]
ejemplos.append({"messages": [
    {"role": "system", "content": "Eres el asistente turistico de \"Cholets Turistico El Alto\", una plataforma de locales y cholets de El Alto, Bolivia. Responde en español, breve y amable."},
    {"role": "user", "content": "¿Qué es un cholet?"},
]})
for e in ejemplos:
    mensajes = e["messages"][:2]
    entrada = tokenizer.apply_chat_template(mensajes, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    salida = model.generate(input_ids=entrada, max_new_tokens=220, temperature=0.2, do_sample=True)
    respuesta = tokenizer.decode(salida[0][entrada.shape[1]:], skip_special_tokens=True)
    print("PREGUNTA:", mensajes[1]["content"])
    print("RESPUESTA:", respuesta)
    print("-" * 80)

## 9. Exportar a GGUF para Ollama (5-10 minutos)
Se guarda cuantizado en **q4_k_m** (~1 GB), el mismo tipo que el modelo original de Ollama.

In [ ]:
import glob, os, shutil

model.save_pretrained_gguf("cholets_qwen", tokenizer, quantization_method="q4_k_m")
gguf = [f for f in glob.glob("**/*.gguf", recursive=True) if "q4_k_m" in f.lower()]
print(gguf)
shutil.copy(gguf[0], "cholets-qwen.gguf")
print(f"Tamaño: {os.path.getsize('cholets-qwen.gguf') / 1e9:.2f} GB")

## 10. Descargar el modelo
**Opción A:** descarga directa (puede tardar unos minutos por el tamaño).
**Opción B (si la A falla):** copiarlo a tu Google Drive y bajarlo desde ahí.

In [ ]:
# Opcion A
files.download("cholets-qwen.gguf")

In [ ]:
# Opcion B: a Google Drive (te pedira permiso para tu Drive)
from google.colab import drive
drive.mount("/content/drive")
shutil.copy("cholets-qwen.gguf", "/content/drive/MyDrive/cholets-qwen.gguf")
print("Listo: MyDrive/cholets-qwen.gguf")

## 11. (Opcional) Guardar el adaptador LoRA para el anexo de la tesis (~70 MB)

In [ ]:
model.save_pretrained("lora_cholets")
tokenizer.save_pretrained("lora_cholets")
shutil.make_archive("lora_cholets", "zip", "lora_cholets")
files.download("lora_cholets.zip")